# Target-Date Feature Validation — DA Load & Weather (D+1)

**Sources:**
- `load_da_hourly.pull()` — DA load forecast (2020+), shifted back 1 day
- `weather_hourly.pull()` — Observed weather (1995+), shifted back 1 day as "perfect forecast" proxy

**Purpose:** Validate target-date (D+1) features that let the model match on "what tomorrow is expected to look like." Cross-validate that shifted features correctly align with reference-date features.

### Checks
1. **Setup & Data Pull** — Pull DA load and weather, build reference + target features
2. **Previous 3 Days** — DA load and weather hourly data for recent days
3. **Completeness** — Date coverage, NaN counts per feature
4. **Cross-Validation** — Verify `tgt_load_daily_avg[D]` == `load_daily_avg[D+1]`
5. **Scatter Plots** — Target vs reference feature relationships
6. **Distributions** — Change features (`tgt_*_change_vs_ref`)
7. **Feature Correlation** — Heatmap of all target features
8. **Validation Summary** — Pass/fail checks

## 1. Setup & Data Pull

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.data import load_da_hourly, weather_hourly
from src.like_day_forecast.features import load_features, weather_features
from src.like_day_forecast.features import target_load_features, target_weather_features
from src.like_day_forecast import configs

HOURS = list(range(1, 25))

In [ ]:
# Pull raw hourly data
df_da_load = load_da_hourly.pull()
df_da_load["date"] = pd.to_datetime(df_da_load["date"])

df_weather = weather_hourly.pull()
df_weather["date"] = pd.to_datetime(df_weather["date"])

print("=== DA Load ===")
print(f"  Shape: {df_da_load.shape}")
print(f"  Date range: {df_da_load['date'].min().date()} to {df_da_load['date'].max().date()}")
print(f"  Unique dates: {df_da_load['date'].nunique():,}")

print("\n=== Weather ===")
print(f"  Shape: {df_weather.shape}")
print(f"  Date range: {df_weather['date'].min().date()} to {df_weather['date'].max().date()}")
print(f"  Unique dates: {df_weather['date'].nunique():,}")

In [ ]:
# Build reference-date features (needed for cross-day deltas)
df_ref_load = load_features.build(df_da_load=df_da_load)
df_ref_weather = weather_features.build(df_weather=df_weather)

# Build target-date features
df_tgt_load = target_load_features.build(df_da_load=df_da_load, df_ref_load_features=df_ref_load)
df_tgt_weather = target_weather_features.build(df_weather=df_weather, df_ref_weather_features=df_ref_weather)

# Convert dates for merging
df_ref_load["date"] = pd.to_datetime(df_ref_load["date"])
df_ref_weather["date"] = pd.to_datetime(df_ref_weather["date"])
df_tgt_load["date"] = pd.to_datetime(df_tgt_load["date"])
df_tgt_weather["date"] = pd.to_datetime(df_tgt_weather["date"])

print("=== Reference Load Features ===")
print(f"  Shape: {df_ref_load.shape}, Columns: {list(df_ref_load.columns)}")
print(f"  Date range: {df_ref_load['date'].min().date()} to {df_ref_load['date'].max().date()}")

print("\n=== Target Load Features ===")
print(f"  Shape: {df_tgt_load.shape}, Columns: {list(df_tgt_load.columns)}")
print(f"  Date range: {df_tgt_load['date'].min().date()} to {df_tgt_load['date'].max().date()}")

print("\n=== Reference Weather Features ===")
print(f"  Shape: {df_ref_weather.shape}, Columns: {list(df_ref_weather.columns)}")
print(f"  Date range: {df_ref_weather['date'].min().date()} to {df_ref_weather['date'].max().date()}")

print("\n=== Target Weather Features ===")
print(f"  Shape: {df_tgt_weather.shape}, Columns: {list(df_tgt_weather.columns)}")
print(f"  Date range: {df_tgt_weather['date'].min().date()} to {df_tgt_weather['date'].max().date()}")

## 2. Previous 3 Days — DA Load & Weather Hourly

Quick visual inspection of the most recent 3 days of raw hourly data for both DA load and weather.

In [ ]:
# Previous 3 days — DA Load hourly
last_3_da = sorted(df_da_load["date"].unique())[-3:]
df_da_last3 = df_da_load[df_da_load["date"].isin(last_3_da)].sort_values(["date", "hour_ending"])
print(f"=== DA Load — Previous 3 Days ===")
print(f"Dates: {[str(d.date()) for d in last_3_da]}\n")
display(df_da_last3.pivot_table(index="hour_ending", columns="date", values="da_load_mw"))

# Previous 3 days — Weather hourly (temp, feels_like_temp)
last_3_wx = sorted(df_weather["date"].unique())[-3:]
df_wx_last3 = df_weather[df_weather["date"].isin(last_3_wx)].sort_values(["date", "hour_ending"])
print(f"\n=== Temperature (°F) — Previous 3 Days ===")
display(df_wx_last3.pivot_table(index="hour_ending", columns="date", values="temp"))

print(f"\n=== Feels-Like Temperature (°F) — Previous 3 Days ===")
display(df_wx_last3.pivot_table(index="hour_ending", columns="date", values="feels_like_temp"))

In [ ]:
# Hourly profiles — DA load and temperature for last 3 days
fig = make_subplots(rows=1, cols=2, subplot_titles=["DA Load (MW)", "Temperature (°F)"])

for d in last_3_da:
    subset = df_da_last3[df_da_last3["date"] == d]
    fig.add_trace(go.Scatter(x=subset["hour_ending"], y=subset["da_load_mw"],
                              mode="lines+markers", name=str(d.date()), showlegend=True),
                  row=1, col=1)

for d in last_3_wx:
    subset = df_wx_last3[df_wx_last3["date"] == d]
    fig.add_trace(go.Scatter(x=subset["hour_ending"], y=subset["temp"],
                              mode="lines+markers", name=str(d.date()), showlegend=False),
                  row=1, col=2)

fig.update_layout(height=400, title_text="Previous 3 Days — DA Load & Temperature Hourly Profiles")
fig.update_xaxes(dtick=2)
fig.show()

## 3. Completeness Checks

Verify target feature date coverage and NaN counts.

In [ ]:
for label, df_tgt, df_ref in [
    ("Load", df_tgt_load, df_ref_load),
    ("Weather", df_tgt_weather, df_ref_weather),
]:
    print(f"=== Target {label} Features ===")
    feat_cols = [c for c in df_tgt.columns if c != "date"]
    print(f"  Features: {feat_cols}")
    print(f"  Rows: {len(df_tgt):,}")
    print(f"  Date range: {df_tgt['date'].min().date()} to {df_tgt['date'].max().date()}")

    # NaN counts per feature
    null_summary = pd.DataFrame({
        "null_count": df_tgt[feat_cols].isnull().sum(),
        "null_pct": (df_tgt[feat_cols].isnull().sum() / len(df_tgt) * 100).round(3),
    })
    print(f"\n  NaN Summary:")
    display(null_summary)

    # Date overlap with reference features
    tgt_dates = set(df_tgt["date"].values)
    ref_dates = set(df_ref["date"].values)
    overlap = tgt_dates & ref_dates
    tgt_only = tgt_dates - ref_dates
    ref_only = ref_dates - tgt_dates
    print(f"\n  Date overlap with reference: {len(overlap):,}")
    print(f"  Target-only dates: {len(tgt_only)} (expected: last date has no D+1)")
    print(f"  Reference-only dates: {len(ref_only)} (expected: first date has no D-1 target)")

    # Descriptive statistics
    print(f"\n  Descriptive Statistics:")
    display(df_tgt[feat_cols].describe().round(2))
    print("\n" + "=" * 60 + "\n")

## 4. Cross-Validation — Shift Identity Check

The critical test: `tgt_load_daily_avg[D]` should equal `load_daily_avg[D+1]`, and similarly for weather. This verifies the 1-day shift is correct.

In [ ]:
# --- Load shift identity: tgt_load_daily_avg[D] == load_daily_avg[D+1] ---
# For target features on date D, the underlying data comes from D+1.
# So tgt_load_daily_avg on date D should equal load_daily_avg on date D+1.

ref_shifted = df_ref_load[["date", "load_daily_avg"]].copy()
ref_shifted["date"] = ref_shifted["date"] - pd.Timedelta(days=1)  # shift ref to align with target
ref_shifted = ref_shifted.rename(columns={"load_daily_avg": "ref_load_avg_d_plus_1"})

load_check = df_tgt_load[["date", "tgt_load_daily_avg"]].merge(ref_shifted, on="date", how="inner")
load_check["match"] = np.isclose(
    load_check["tgt_load_daily_avg"], load_check["ref_load_avg_d_plus_1"], rtol=1e-6
)

load_identity_pass = load_check["match"].all()
n_load_checked = len(load_check)

print(f"=== Load Shift Identity Check ===")
print(f"  Compared {n_load_checked:,} date pairs")
print(f"  All match: {load_identity_pass}")
if not load_identity_pass:
    mismatches = load_check[~load_check["match"]]
    print(f"  Mismatches ({len(mismatches)}):")
    display(mismatches.head(10))
else:
    print(f"  Sample (first 5):")
    display(load_check.head())

In [ ]:
# --- Weather shift identity: tgt_temp_daily_avg[D] == temp_daily_avg[D+1] ---
ref_wx_shifted = df_ref_weather[["date", "temp_daily_avg"]].copy()
ref_wx_shifted["date"] = ref_wx_shifted["date"] - pd.Timedelta(days=1)
ref_wx_shifted = ref_wx_shifted.rename(columns={"temp_daily_avg": "ref_temp_avg_d_plus_1"})

wx_check = df_tgt_weather[["date", "tgt_temp_daily_avg"]].merge(ref_wx_shifted, on="date", how="inner")
wx_check["match"] = np.isclose(
    wx_check["tgt_temp_daily_avg"], wx_check["ref_temp_avg_d_plus_1"], rtol=1e-6
)

wx_identity_pass = wx_check["match"].all()
n_wx_checked = len(wx_check)

print(f"=== Weather Shift Identity Check ===")
print(f"  Compared {n_wx_checked:,} date pairs")
print(f"  All match: {wx_identity_pass}")
if not wx_identity_pass:
    mismatches = wx_check[~wx_check["match"]]
    print(f"  Mismatches ({len(mismatches)}):")
    display(mismatches.head(10))
else:
    print(f"  Sample (first 5):")
    display(wx_check.head())

In [ ]:
# --- Cross-day delta verification ---
# tgt_load_change_vs_ref[D] = tgt_load_daily_avg[D] - load_daily_avg[D]
# Verify this holds for all dates with both features present

delta_check = df_tgt_load[["date", "tgt_load_daily_avg", "tgt_load_change_vs_ref"]].merge(
    df_ref_load[["date", "load_daily_avg"]], on="date", how="inner"
).dropna()

delta_check["expected"] = delta_check["tgt_load_daily_avg"] - delta_check["load_daily_avg"]
delta_check["delta_match"] = np.isclose(
    delta_check["tgt_load_change_vs_ref"], delta_check["expected"], atol=0.01
)

load_delta_pass = delta_check["delta_match"].all()
print(f"=== Load Change vs Ref Formula Check ===")
print(f"  Checked {len(delta_check):,} rows")
print(f"  Formula verified: {load_delta_pass}")

# Same for weather
wx_delta_check = df_tgt_weather[["date", "tgt_temp_daily_avg"]].copy()
if "tgt_temp_change_vs_ref" in df_tgt_weather.columns:
    wx_delta_check["tgt_temp_change_vs_ref"] = df_tgt_weather["tgt_temp_change_vs_ref"]
    wx_delta_check = wx_delta_check.merge(
        df_ref_weather[["date", "temp_daily_avg"]], on="date", how="inner"
    ).dropna()
    wx_delta_check["expected"] = wx_delta_check["tgt_temp_daily_avg"] - wx_delta_check["temp_daily_avg"]
    wx_delta_check["delta_match"] = np.isclose(
        wx_delta_check["tgt_temp_change_vs_ref"], wx_delta_check["expected"], atol=0.01
    )
    wx_delta_pass = wx_delta_check["delta_match"].all()
    print(f"\n=== Weather Change vs Ref Formula Check ===")
    print(f"  Checked {len(wx_delta_check):,} rows")
    print(f"  Formula verified: {wx_delta_pass}")
else:
    wx_delta_pass = True
    print("\n  (tgt_temp_change_vs_ref not present — skipping)")

## 5. Scatter Plots — Target vs Reference Features

Verify that target features correlate with reference features and explore the relationship between D and D+1 conditions.

In [ ]:
# Merge reference + target load features for scatter plots
load_merged = df_ref_load.merge(df_tgt_load, on="date", how="inner")

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Avg Load: Target vs Ref", "Peak Load: Target vs Ref", "Peak Ratio: Target vs Ref"
])

pairs = [
    ("load_daily_avg", "tgt_load_daily_avg"),
    ("load_daily_peak", "tgt_load_daily_peak"),
    ("load_peak_ratio", "tgt_load_peak_ratio"),
]

for i, (ref_col, tgt_col) in enumerate(pairs):
    if ref_col in load_merged.columns and tgt_col in load_merged.columns:
        corr = load_merged[[ref_col, tgt_col]].corr().iloc[0, 1]
        fig.add_trace(go.Scatter(
            x=load_merged[ref_col], y=load_merged[tgt_col],
            mode="markers", marker=dict(size=2, opacity=0.3, color="steelblue"),
            name=f"r={corr:.3f}", showlegend=True,
        ), row=1, col=i+1)
        # 1:1 line
        xy_min = min(load_merged[ref_col].min(), load_merged[tgt_col].min())
        xy_max = max(load_merged[ref_col].max(), load_merged[tgt_col].max())
        fig.add_trace(go.Scatter(
            x=[xy_min, xy_max], y=[xy_min, xy_max],
            mode="lines", line=dict(color="yellow", dash="dash", width=1),
            showlegend=False,
        ), row=1, col=i+1)

fig.update_layout(height=400, title_text="Target vs Reference Load Features")
fig.update_xaxes(title_text="Reference (D)")
fig.update_yaxes(title_text="Target (D+1)")
fig.show()

In [ ]:
# Merge reference + target weather features for scatter plots
wx_merged = df_ref_weather.merge(df_tgt_weather, on="date", how="inner")

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Avg Temp: Target vs Ref", "HDD: Target vs Ref", "CDD: Target vs Ref"
])

wx_pairs = [
    ("temp_daily_avg", "tgt_temp_daily_avg"),
    ("hdd", "tgt_hdd"),
    ("cdd", "tgt_cdd"),
]

colors = ["steelblue", "steelblue", "firebrick"]

for i, ((ref_col, tgt_col), color) in enumerate(zip(wx_pairs, colors)):
    if ref_col in wx_merged.columns and tgt_col in wx_merged.columns:
        corr = wx_merged[[ref_col, tgt_col]].corr().iloc[0, 1]
        fig.add_trace(go.Scatter(
            x=wx_merged[ref_col], y=wx_merged[tgt_col],
            mode="markers", marker=dict(size=2, opacity=0.3, color=color),
            name=f"r={corr:.3f}", showlegend=True,
        ), row=1, col=i+1)
        xy_min = min(wx_merged[ref_col].min(), wx_merged[tgt_col].min())
        xy_max = max(wx_merged[ref_col].max(), wx_merged[tgt_col].max())
        fig.add_trace(go.Scatter(
            x=[xy_min, xy_max], y=[xy_min, xy_max],
            mode="lines", line=dict(color="yellow", dash="dash", width=1),
            showlegend=False,
        ), row=1, col=i+1)

fig.update_layout(height=400, title_text="Target vs Reference Weather Features")
fig.update_xaxes(title_text="Reference (D)")
fig.update_yaxes(title_text="Target (D+1)")
fig.show()

## 6. Distributions of Change Features

The `tgt_*_change_vs_ref` features capture the transition signal between D and D+1. They should be centered near 0 with tails reflecting weather fronts and demand swings.

In [ ]:
# Histograms for change features
change_cols = []
change_data = {}

if "tgt_load_change_vs_ref" in df_tgt_load.columns:
    change_cols.append("tgt_load_change_vs_ref")
    change_data["tgt_load_change_vs_ref"] = df_tgt_load["tgt_load_change_vs_ref"].dropna()

if "tgt_temp_change_vs_ref" in df_tgt_weather.columns:
    change_cols.append("tgt_temp_change_vs_ref")
    change_data["tgt_temp_change_vs_ref"] = df_tgt_weather["tgt_temp_change_vs_ref"].dropna()

fig = make_subplots(rows=1, cols=len(change_cols), subplot_titles=change_cols)

for i, col in enumerate(change_cols):
    series = change_data[col]
    fig.add_trace(go.Histogram(
        x=series, nbinsx=60, name=col, showlegend=False,
    ), row=1, col=i+1)
    fig.add_vline(x=0, line_dash="dash", line_color="yellow", row=1, col=i+1)

fig.update_layout(height=400, title_text="Distribution of D+1 vs D Change Features")
fig.show()

# Stats
for col in change_cols:
    s = change_data[col]
    print(f"{col}:")
    print(f"  mean={s.mean():.2f}, std={s.std():.2f}, "
          f"min={s.min():.1f}, max={s.max():.1f}, "
          f"|change| > 2*std: {(s.abs() > 2*s.std()).sum()}")
    print()

## 7. Feature Correlation Matrix

Heatmap of all target features (load + weather combined).

In [ ]:
# Combine all target features
all_tgt = df_tgt_load.merge(df_tgt_weather, on="date", how="outer")
feat_cols = [c for c in all_tgt.columns if c != "date"]
corr = all_tgt[feat_cols].corr()

fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Target Feature Correlation Matrix (Load + Weather)", aspect="auto")
fig.update_layout(height=700, width=800)
fig.show()

# Flag highly correlated pairs
print("Highly correlated target feature pairs (|r| > 0.90):")
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.90:
            print(f"  {feat_cols[i]} <-> {feat_cols[j]}: r = {r:.3f}")

## 8. Validation Summary

In [ ]:
checks = [
    ("Target load features have 6 columns (excl date)",
     len([c for c in df_tgt_load.columns if c != "date"]) == 6,
     f"{len([c for c in df_tgt_load.columns if c != 'date'])} columns"),

    ("Target weather features have 7 columns (excl date)",
     len([c for c in df_tgt_weather.columns if c != "date"]) == 7,
     f"{len([c for c in df_tgt_weather.columns if c != 'date'])} columns"),

    ("Target load date range starts 2020 or earlier",
     df_tgt_load["date"].min().year <= 2020,
     f"Starts {df_tgt_load['date'].min().date()}"),

    ("Target weather date range starts 1995 or earlier",
     df_tgt_weather["date"].min().year <= 1995,
     f"Starts {df_tgt_weather['date'].min().date()}"),

    ("Load shift identity: tgt_load_daily_avg[D] == load_daily_avg[D+1]",
     load_identity_pass,
     f"Checked {n_load_checked:,} pairs"),

    ("Weather shift identity: tgt_temp_daily_avg[D] == temp_daily_avg[D+1]",
     wx_identity_pass,
     f"Checked {n_wx_checked:,} pairs"),

    ("Load change delta formula verified",
     load_delta_pass,
     f"Checked {len(delta_check):,} rows"),

    ("Weather change delta formula verified",
     wx_delta_pass,
     f"Checked {len(wx_delta_check):,} rows"),

    ("No unexpected NaN in tgt_load_daily_avg",
     df_tgt_load["tgt_load_daily_avg"].isnull().sum() == 0,
     f"{df_tgt_load['tgt_load_daily_avg'].isnull().sum()} nulls"),

    ("No unexpected NaN in tgt_temp_daily_avg",
     df_tgt_weather["tgt_temp_daily_avg"].isnull().sum() == 0,
     f"{df_tgt_weather['tgt_temp_daily_avg'].isnull().sum()} nulls"),

    ("Target load peak ratio > 1.0 (all rows)",
     (df_tgt_load["tgt_load_peak_ratio"].dropna() > 1.0).all(),
     f"Min: {df_tgt_load['tgt_load_peak_ratio'].min():.3f}"),

    ("Target HDD/CDD not both positive on same day",
     ((df_tgt_weather["tgt_hdd"] > 0) & (df_tgt_weather["tgt_cdd"] > 0)).sum() == 0,
     f"{((df_tgt_weather['tgt_hdd'] > 0) & (df_tgt_weather['tgt_cdd'] > 0)).sum()} violations"),
]

print("=" * 80)
print("  TARGET FEATURES VALIDATION SUMMARY")
print("=" * 80)
all_pass = True
for name, passed, detail in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {name}")
    print(f"         {detail}")
print("=" * 80)
print(f"  {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 80)